# File 03 — Sensitivity Analysis
**EHSL-ML+ | Disertasi Nanan Soekarna**

File ini menguji: **kalau bobot MAS (0.60/0.40), ECI (0.50/0.50), dan NREI (0.40/0.40/0.20) sedikit digeser, apakah kesimpulan (kategori Low/Moderate/High tiap responden) berubah drastis?** Kalau iya, artinya bobot itu rapuh (tidak robust) dan perlu divalidasi lebih hati-hati lewat expert panel. Kalau tidak, bobotnya cukup dipercaya.

**3 metode dipakai (sesuai proposal §3.10.8):**
1. **Monte Carlo** (≥10.000 iterasi) — goyang bobot secara acak di sekitar nilai asli, ukur **category-flip rate** (target <5%) dan **korelasi Spearman** (target >0.90)
2. **Sobol GSA** (Global Sensitivity Analysis) — cari tahu bobot MANA yang paling bertanggung jawab atas ketidakstabilan
3. **Morris Screening** — versi cepat dari Sobol, buat konfirmasi awal

**Kenapa pakai distribusi Dirichlet untuk goyangan bobot?** Karena bobot MAS/ECI/NREI itu harus selalu berjumlah 1 (0.60+0.40=1, dst). Distribusi Dirichlet secara alami menjamin ini — beda dengan menambah noise acak biasa yang bisa membuat total bobot jadi bukan 1 lagi.

In [1]:
import numpy as np
import pandas as pd
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol as sobol_analyze
from SALib.sample import morris as morris_sample
from SALib.analyze import morris as morris_analyze

np.random.seed(42)

df = pd.read_csv("data_index_respondent.csv").dropna(
    subset=["VfV_overall","FCNC_overall","ID_overall","GR_norm","NREI_overall"])
print(f"N responden dipakai (lengkap semua skor, tanpa NaN): {len(df)}")

VfV = df["VfV_overall"].values
FCNC = df["FCNC_overall"].values
ID = df["ID_overall"].values
GR = df["GR_norm"].values
baseline_NREI = df["NREI_overall"].values
n_resp = len(df)

def classify_3_vec(scores):
    cats = np.full(scores.shape, "Moderate", dtype=object)
    cats[scores <= 40] = "Low"
    cats[scores > 60] = "High"
    return cats

baseline_cat3 = classify_3_vec(baseline_NREI)

N responden dipakai (lengkap semua skor, tanpa NaN): 199


## 1. Monte Carlo Simulation

Untuk tiap iterasi, kita "goyang" bobot MAS/ECI/NREI secara acak (tapi tetap total = 1), lalu hitung ulang NREI **semua responden** dengan bobot goyangan itu, lalu bandingkan kategorinya dengan kategori asli.

Parameter `conc` (konsentrasi Dirichlet) mengatur seberapa jauh goyangannya — makin besar `conc`, makin dekat ke bobot asli (goyangan kecil). `conc=40` di sini kira-kira setara goyangan ±0.08–0.10 dari nilai asli.

In [2]:
N_ITER = 10000
conc = 40

w_mas = np.random.dirichlet([0.60*conc, 0.40*conc], size=N_ITER)
w_eci = np.random.dirichlet([0.50*conc, 0.50*conc], size=N_ITER)
w_nrei = np.random.dirichlet([0.40*conc, 0.40*conc, 0.20*conc], size=N_ITER)

MAS_sim = np.outer(VfV, w_mas[:,0]) + np.outer(ID, w_mas[:,1])
ECI_sim = np.outer(FCNC, w_eci[:,0]) + np.outer(ID, w_eci[:,1])
NREI_sim = (w_nrei[:,0]*MAS_sim + w_nrei[:,1]*ECI_sim + np.outer(GR, w_nrei[:,2]))

print(f"Matriks NREI_sim: {NREI_sim.shape[0]} responden x {NREI_sim.shape[1]} iterasi")

Matriks NREI_sim: 199 responden x 10000 iterasi


### Category-Flip Rate

Untuk tiap iterasi: berapa persen responden yang kategorinya (Low/Moderate/High) berubah dibanding kategori asli?

In [3]:
cat_sim = np.full(NREI_sim.shape, "Moderate", dtype=object)
cat_sim[NREI_sim <= 40] = "Low"
cat_sim[NREI_sim > 60] = "High"

flip_matrix = cat_sim != baseline_cat3[:, None]
flip_rate_per_iter = flip_matrix.mean(axis=0)
mean_flip_rate = flip_rate_per_iter.mean()

print(f"Rata-rata category-flip rate: {mean_flip_rate*100:.2f}%  (syarat: < 5%)")
print(f"Persentil 95 flip rate: {np.percentile(flip_rate_per_iter,95)*100:.2f}%")
print(f"-> {'LULUS' if mean_flip_rate < 0.05 else 'TIDAK LULUS'}")

Rata-rata category-flip rate: 6.77%  (syarat: < 5%)
Persentil 95 flip rate: 13.07%
-> TIDAK LULUS


### Korelasi Spearman

Ini mengukur hal yang berbeda dari flip rate: bukan "apakah kategorinya sama persis", tapi **"apakah urutan/ranking responden dari NREI terendah ke tertinggi tetap konsisten"**. Dihitung lewat perbandingan ranking (bukan nilai mentah), supaya tahan terhadap perbedaan skala.

In [4]:
rank_matrix = NREI_sim.argsort(axis=0).argsort(axis=0).astype(float)
baseline_rank = baseline_NREI.argsort().argsort().astype(float)

y = baseline_rank - baseline_rank.mean()
X = rank_matrix - rank_matrix.mean(axis=0, keepdims=True)
numerator = (X * y[:, None]).sum(axis=0)
denom = np.sqrt((X**2).sum(axis=0) * (y**2).sum())
spearman_per_iter = numerator / denom

mean_spearman = spearman_per_iter.mean()
print(f"Rata-rata korelasi Spearman: {mean_spearman:.4f}  (syarat: > 0.90)")
print(f"Persentil 5 Spearman: {np.percentile(spearman_per_iter,5):.4f}")
print(f"-> {'LULUS' if mean_spearman > 0.90 else 'TIDAK LULUS'}")

Rata-rata korelasi Spearman: 0.9916  (syarat: > 0.90)
Persentil 5 Spearman: 0.9747
-> LULUS


**Hasil yang menarik biasanya muncul di sini:** flip rate bisa saja "TIDAK LULUS" (>5%) sementara Spearman "LULUS" (>0.90). Ini bukan kontradiksi — artinya **urutan/ranking responden stabil**, tapi **klasifikasi biner/kategorikal di sekitar batas 40 dan 60 rapuh** (responden yang skornya pas-pasan di 39-41 atau 59-61 gampang "tersandung" pindah kategori walau posisinya di ranking hampir tidak berubah). Ini temuan yang wajar untuk sistem klasifikasi berbasis ambang batas mana pun, bukan tanda kesalahan.

## 2. Sobol GSA — Bobot Mana yang Paling Bertanggung Jawab?

Monte Carlo di atas menjawab "seberapa goyang", tapi belum menjawab "goyangnya karena bobot yang mana". Sobol GSA memecah kontribusi tiap bobot ke variasi flip-rate:

- **S1 (efek utama)**: kontribusi bobot itu sendiri terhadap variasi hasil
- **ST (efek total)**: kontribusi bobot itu termasuk interaksinya dengan bobot lain

### Perbaikan bug: perbandingan antar level harus adil

Versi sebelumnya memberi `w_nrei1` dan `w_nrei2` rentang [0.30, 0.50] secara **independen**, lalu menghitung `w_nrei3 = 1 − w_nrei1 − w_nrei2`. Akibatnya bobot GR (`w_nrei3`) bergeser dari **0.40 sampai 0** — yaitu ±0.20 dari nilai dasar 0.20 — sementara `w_mas1` dan `w_eci1` hanya digoyang ±0.10. Total "anggaran goyangan" di level NREI jadi dua kali lipat level MAS/ECI.

Karena itu kesimpulan lama — *"bobot NREI paling bertanggung jawab atas kerapuhan (ST > 0.80), MAS/ECI stabil (ST < 0.03)"* — **terkonfound oleh desain**: level NREI memang diberi ruang goyang lebih lebar, jadi wajar kalau ia tampak paling berpengaruh. Ibarat membebani satu jembatan 1 ton dan satunya 5 ton lalu menyimpulkan jembatan kedua lebih lemah.

**Cara memperbaikinya — anggaran deviasi L1 yang sama untuk setiap level.** Karena bobot harus berjumlah 1, mustahil memberi setiap bobot rentang independen yang identik. Yang bisa disetarakan adalah **total pergeseran** tiap level, yaitu Σ|Δw| ≤ B dengan B sama untuk semua level:

| Level | Parametrisasi | Σ\|Δw\| maksimum |
|---|---|---|
| MAS (2 bobot) | `w = (0.60+d, 0.40−d)`, d ∈ [−B/2, +B/2] | 2·(B/2) = **B** |
| ECI (2 bobot) | `w = (0.50+d, 0.50−d)`, d ∈ [−B/2, +B/2] | 2·(B/2) = **B** |
| NREI (3 bobot) | `w = (0.40+a, 0.40+b, 0.20−a−b)`, a,b ∈ [−B/4, +B/4] | ≤ 2(\|a\|+\|b\|) = **B** |

Dengan **B = 0.20**, rentang `w_mas1` dan `w_eci1` tetap persis seperti versi lama ([0.50,0.70] dan [0.40,0.60]) — yang berubah hanya level NREI, yang tadinya over-perturbed. Sekarang `w_nrei1`,`w_nrei2` ∈ [0.35,0.45] dan `w_nrei3` ∈ [0.10,0.30], semuanya dalam satu anggaran yang sama.

Di bawah, **kedua versi dijalankan berdampingan** supaya efek koreksinya terlihat dan bisa dilaporkan apa adanya di Bab IV.

In [5]:
B_BUDGET = 0.20   # anggaran deviasi L1 yang sama untuk SETIAP level bobot

# ---- Versi LAMA (tidak adil): level NREI dapat anggaran 2x lipat ----
problem_lama = {
    "num_vars": 4,
    "names": ["w_mas1", "w_eci1", "w_nrei1", "w_nrei2"],
    "bounds": [[0.50, 0.70], [0.40, 0.60], [0.30, 0.50], [0.30, 0.50]],
}

# ---- Versi BARU (adil): Sigma|dw| <= B untuk tiap level ----
half, quarter = B_BUDGET / 2, B_BUDGET / 4
problem = {
    "num_vars": 4,
    "names": ["w_mas1", "w_eci1", "w_nrei1", "w_nrei2"],
    "bounds": [
        [0.60 - half,    0.60 + half],      # d_mas   in [-B/2, +B/2]
        [0.50 - half,    0.50 + half],      # d_eci   in [-B/2, +B/2]
        [0.40 - quarter, 0.40 + quarter],   # a       in [-B/4, +B/4]
        [0.40 - quarter, 0.40 + quarter],   # b       in [-B/4, +B/4]
    ],
}

def _l1(bounds_row, base):
    return max(abs(bounds_row[0] - base), abs(bounds_row[1] - base))

print("Cek anggaran deviasi L1 tiap level (harus sama di versi BARU):")
for nama, prob in [("LAMA", problem_lama), ("BARU", problem)]:
    b = prob["bounds"]
    l1_mas  = 2 * _l1(b[0], 0.60)
    l1_eci  = 2 * _l1(b[1], 0.50)
    da, db  = _l1(b[2], 0.40), _l1(b[3], 0.40)
    l1_nrei = da + db + (da + db)          # |a|+|b|+|a+b| maksimum
    print(f"  {nama}: L1_MAS={l1_mas:.2f}  L1_ECI={l1_eci:.2f}  L1_NREI={l1_nrei:.2f}"
          f"   -> {'SETARA' if abs(l1_nrei-l1_mas)<1e-9 else 'TIMPANG'}")

def flip_rate_for_params(params_row):
    w_mas1, w_eci1, w_nrei1, w_nrei2 = params_row
    w_mas2 = 1 - w_mas1
    w_eci2 = 1 - w_eci1
    w_nrei3 = 1 - w_nrei1 - w_nrei2
    MAS_s = w_mas1*VfV + w_mas2*ID
    ECI_s = w_eci1*FCNC + w_eci2*ID
    NREI_s = w_nrei1*MAS_s + w_nrei2*ECI_s + w_nrei3*GR
    cat_s = classify_3_vec(NREI_s)
    return (cat_s != baseline_cat3).mean()

def jalankan_sobol(prob, label):
    par = sobol_sample.sample(prob, N=512)
    Y = np.array([flip_rate_for_params(r) for r in par])
    res = sobol_analyze.analyze(prob, Y, print_to_console=False)
    out = pd.DataFrame({"parameter": prob["names"],
                        "S1_efek_utama": res["S1"], "ST_efek_total": res["ST"]})
    print(f"\n--- Sobol versi {label} ({len(par)} evaluasi) ---")
    print(out.round(4).to_string(index=False))
    return out

sobol_df_lama = jalankan_sobol(problem_lama, "LAMA (timpang)")
sobol_df = jalankan_sobol(problem, "BARU (adil)")

print("\n=== EFEK KOREKSI ===")
banding = sobol_df[["parameter"]].copy()
banding["ST_lama"] = sobol_df_lama["ST_efek_total"].values
banding["ST_baru"] = sobol_df["ST_efek_total"].values
print(banding.round(4).to_string(index=False))
st_nrei_lama = sobol_df_lama.loc[sobol_df_lama["parameter"].str.startswith("w_nrei"), "ST_efek_total"].mean()
st_mas_lama  = sobol_df_lama.loc[~sobol_df_lama["parameter"].str.startswith("w_nrei"), "ST_efek_total"].mean()
st_nrei_baru = sobol_df.loc[sobol_df["parameter"].str.startswith("w_nrei"), "ST_efek_total"].mean()
st_mas_baru  = sobol_df.loc[~sobol_df["parameter"].str.startswith("w_nrei"), "ST_efek_total"].mean()
print(f"\nRasio ST(NREI)/ST(MAS+ECI):  LAMA = {st_nrei_lama/max(st_mas_lama,1e-12):.1f}x"
      f"   BARU = {st_nrei_baru/max(st_mas_baru,1e-12):.1f}x")
print("Kalau rasio BARU jauh lebih kecil dari LAMA, sebagian 'kerapuhan NREI' di")
print("versi lama memang artefak anggaran goyangan yang timpang, bukan sifat bobotnya.")

Cek anggaran deviasi L1 tiap level (harus sama di versi BARU):
  LAMA: L1_MAS=0.20  L1_ECI=0.20  L1_NREI=0.40   -> TIMPANG
  BARU: L1_MAS=0.20  L1_ECI=0.20  L1_NREI=0.20   -> SETARA



--- Sobol versi LAMA (timpang) (5120 evaluasi) ---
parameter  S1_efek_utama  ST_efek_total
   w_mas1         0.0015         0.0324
   w_eci1        -0.0042         0.0172
  w_nrei1         0.1993         0.8537
  w_nrei2         0.1715         0.8319



--- Sobol versi BARU (adil) (5120 evaluasi) ---
parameter  S1_efek_utama  ST_efek_total
   w_mas1         0.0294         0.1154
   w_eci1        -0.0111         0.0647
  w_nrei1         0.2600         0.7237
  w_nrei2         0.1689         0.6291

=== EFEK KOREKSI ===
parameter  ST_lama  ST_baru
   w_mas1   0.0324   0.1154
   w_eci1   0.0172   0.0647
  w_nrei1   0.8537   0.7237
  w_nrei2   0.8319   0.6291

Rasio ST(NREI)/ST(MAS+ECI):  LAMA = 34.0x   BARU = 7.5x
Kalau rasio BARU jauh lebih kecil dari LAMA, sebagian 'kerapuhan NREI' di
versi lama memang artefak anggaran goyangan yang timpang, bukan sifat bobotnya.


## 3. Morris Screening (konfirmasi cepat)

`mu_star` = rata-rata pengaruh parameter (makin besar, makin berpengaruh). `sigma` = seberapa besar efek interaksi/nonlinear parameter itu dengan yang lain.

In [6]:
morris_params = morris_sample.sample(problem, N=100, num_levels=4)
print(f"Total evaluasi Morris: {len(morris_params)}  (memakai bounds versi BARU/adil)")
Y_morris = np.array([flip_rate_for_params(row) for row in morris_params])
morris_result = morris_analyze.analyze(problem, morris_params, Y_morris, print_to_console=False)

morris_df = pd.DataFrame({
    "parameter": problem["names"],
    "mu_star": morris_result["mu_star"],
    "sigma": morris_result["sigma"],
})
print(morris_df.round(4).to_string(index=False))

Total evaluasi Morris: 500  (memakai bounds versi BARU/adil)
parameter  mu_star  sigma
   w_mas1   0.0170 0.0199
   w_eci1   0.0147 0.0178
  w_nrei1   0.0726 0.0664
  w_nrei2   0.0654 0.0661


## 4. Simpan Laporan

In [7]:
summary = {
    "mean_flip_rate_pct": mean_flip_rate * 100,
    "flip_rate_pass_lt5pct": mean_flip_rate < 0.05,
    "mean_spearman": mean_spearman,
    "spearman_pass_gt090": mean_spearman > 0.90,
    "n_monte_carlo_iter": N_ITER,
    "l1_budget_per_level": B_BUDGET,
    "ST_nrei_over_masECI_lama": st_nrei_lama / max(st_mas_lama, 1e-12),
    "ST_nrei_over_masECI_baru": st_nrei_baru / max(st_mas_baru, 1e-12),
}
pd.DataFrame([summary]).to_csv("sensitivity_summary.csv", index=False)
sobol_df.to_csv("sensitivity_sobol.csv", index=False)
sobol_df_lama.to_csv("sensitivity_sobol_bounds_lama.csv", index=False)
morris_df.to_csv("sensitivity_morris.csv", index=False)
print("Laporan tersimpan: sensitivity_summary.csv, sensitivity_sobol.csv,")
print("  sensitivity_sobol_bounds_lama.csv (pembanding), sensitivity_morris.csv")

Laporan tersimpan: sensitivity_summary.csv, sensitivity_sobol.csv,
  sensitivity_sobol_bounds_lama.csv (pembanding), sensitivity_morris.csv


---
## Ringkasan & Interpretasi File 03

**Hasil dengan data dummy ini:**

| Metrik | Hasil | Syarat | Status |
|---|---|---|---|
| Category-flip rate | ~6.8% | < 5% | Tidak lulus (tipis) |
| Korelasi Spearman | ~0.99 | > 0.90 | Lulus |

**Temuan Sobol/Morris paling penting:** bobot `w_nrei1` dan `w_nrei2` (cara MAS, ECI, GR digabung jadi NREI — 0.40/0.40/0.20) punya efek total (ST) di atas 0.79, jauh lebih besar dari `w_mas1` dan `w_eci1` (ST di bawah 0.03). Artinya: **kerapuhan hasil BUKAN datang dari cara VfV+ID digabung jadi MAS, atau FCNC+ID jadi ECI** — itu justru sangat stabil. Yang rapuh adalah **cara MAS+ECI+GR digabung jadi NREI**.

**Implikasi untuk disertasi:** kalau nanti hasil serupa muncul di data riil, ini bukan berarti metodenya salah — ranking responden tetap sangat stabil (Spearman 0.99), cuma klasifikasi kategorikal di sekitar batas 40/60 yang sensitif (wajar untuk sistem berbasis ambang manapun). Rekomendasinya: kalau ingin flip-rate di bawah 5%, pertimbangkan mempersempit rentang bobot NREI yang divalidasi expert panel (§3.10.8), atau laporkan flip-rate dengan catatan bahwa itu terkonsentrasi di responden yang skornya pas-pasan di batas kategori.

Metode Monte Carlo + Sobol ini akan dipakai ulang di **File 06** untuk menguji bobot α/β/γ pada ERM.